<h1>
  <font face="Arial" color="#FFE3EC">
    <b>Autonomous workflow engine</b>
  </font>
</h1>

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>AI-Driven Task Extraction, Dependency Mapping, and Audit Tracking</b>
  </font>
</h3>

This system uses Large Language Models to automatically extract action items from meeting transcripts, assign owners, map dependencies and simulate a self-executing workflow.

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>System Architecture</b>
  </font>
</h3>

<h5>
  <font face="Arial" color="#FFE3EC">
    <b>The engine operates on a tri-agent architecture to ensure data integrity and explainability:</b>
  </font>
</h5>

* **LLM**: Extracts raw action items and identifies owners.

* **Logic**: Maps dependencies and initiates the "In Progress" state.

* **Validator**: Verifies task completion and updates the Audit Trail.

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>Environment Setup</b>
  </font>
</h3>

**We need to install the necessary libraries:**

`Groq`: To access the high-speed Llama-3 AI model.

`Gradio`: To create a user-friendly web interface.

`Pandas`: To organize our tasks and logs into clean tables.

In [ ]:
!pip install groq gradio pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.8 MB/s eta 0:00:00


<h3>
  <font face="Arial" color="#FFE3EC">
    <b>Core Dependencies & API Configuration</b>
  </font>
</h3>

We import the tools needed for data handling (json, pandas) and time tracking and we also securely fetch the Groq API Key from your Colab Secrets

In [ ]:
import json
import pandas as pd
import gradio as gr
from datetime import datetime
from groq import Groq
from google.colab import userdata

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>AI Intelligence Layer</b>
  </font>
</h3>

* This section establishes the connection to the AI.
* The `call_intelligence` function sends our meeting notes to the Llama-3 model and asks it to return a structured **JSON** list of tasks.
* JSON ensures the computer can read the AI's "thoughts" without errors.

In [ ]:
try:
    api_key = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=api_key)
except Exception:
    print("API Key Error: Ensure 'GROQ_API_KEY' is in Colab Secrets.")

def call_intelligence(prompt):
    # Sends prompt to AI and forces a structured JSON response
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception:
        return {"decisions": []}

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>The Workflow Engine Logic</b>
  </font>
</h3>

<h4>
  <font face="Arial" color="#FFE3EC">
    <b>This is the "brain" of the project</b>
  </font>
</h4>

**The MeetingIntelligenceSystem class:**

* **Extracts:** Identifies who needs to do what.

* **Links:** Finds dependencies.

* **Simulates**: Moves tasks from "Pending" to "Resolved" while providing a visual progress bar.

* **Audits:** Keeps a log of every action the "AI Agents" take.

In [ ]:
class MeetingIntelligenceSystem:
    def __init__(self):
        self.registry = []     # Stores all tasks
        self.audit_trail = []  # Stores history of AI actions

    def get_progress_bar(self, pct):
        # Creates a visual █░░ bar
        filled = int(pct / 10)
        bar = "█" * filled + "░" * (10 - filled)
        return f"{bar} {pct}%"

    def process_cycle(self, meeting_text):
        try:
            if meeting_text and meeting_text.strip():
                # Prepare history so AI knows what happened previously
                history = "\n".join([f"ID {t['ID']}: {t['Decision']}" for t in self.registry])

                # AI Instruction: Turn text into structured data
                prompt = f"""
                Analyze this transcript: '{meeting_text}'
                Existing Registry: {history if history else "Empty"}

                LOGIC RULES:
                1. Extract every unique action item.
                2. If a task says "after", "once finished", or "then", identify the 'dep_id'.
                3. SEQUENCING: If Task B follows Task A, Task B's 'dep_id' MUST be Task A's ID.
                4. Create a chain: e.g., James(ID:1) -> Linda(ID:2) -> Mark(ID:3).

                Return ONLY JSON:
                {{
                  "decisions": [
                    {{"decision": "string", "owner": "string", "priority": "High/Med/Low", "dep_id": int_or_null}}
                  ]
                }}
                """
                data = call_intelligence(prompt)

                # Save new tasks to the registry
                for d in data.get("decisions", []):
                    new_id = len(self.registry) + 1
                    self.registry.append({
                        "ID": new_id,
                        "Decision": d.get("decision", "Action Item"),
                        "Owner": d.get("owner") or "Unassigned",
                        "Priority": d.get("priority", "Med"),
                        "Status": "⚪ Pending",
                        "Progress": 0,
                        "Dependency": d.get("dep_id")
                    })
                    self.audit_trail.append({"Time": datetime.now().strftime("%H:%M"), "Agent": "🧠 Planner", "Action": f"Indexed Decision #{new_id}"})

            # Cycle through tasks to update progress and handle dependencies
            for _ in range(2):
                for item in self.registry:
                    dep_id = item.get("Dependency")
                    is_blocked = False

                    # Check if the 'parent' task is finished
                    if dep_id is not None:
                        try:

                            parent = next((x for x in self.registry if x["ID"] == int(dep_id)), None)
                            if parent and parent["Progress"] < 100:
                                is_blocked = True
                                item["Status"] = f"🔴 Stalled (ID:{dep_id})"
                        except: pass

                    if not is_blocked:
                        if "Pending" in item["Status"] or "Stalled" in item["Status"]:
                            item["Status"] = "🔵 In Progress"
                            item["Progress"] = 20
                            self.audit_trail.append({"Time": datetime.now().strftime("%H:%M"), "Agent": "⚙️ Executor", "Action": f"Initiated Task #{item['ID']}"})
                        elif item["Status"] == "🔵 In Progress":
                            item["Progress"] = min(item["Progress"] + 40, 100)
                            if item["Progress"] == 100:
                                item["Status"] = "🟢 Resolved"
                                self.audit_trail.append({"Time": datetime.now().strftime("%H:%M"), "Agent": "🏁 Auditor", "Action": f"Verified Task #{item['ID']}"})


            # Format data for display
            df_tasks = pd.DataFrame(self.registry).copy() if self.registry else pd.DataFrame(columns=["ID", "Decision", "Owner", "Status", "Progress"])
            if not df_tasks.empty:
                df_tasks["Progress"] = df_tasks["Progress"].apply(self.get_progress_bar)
                df_tasks = df_tasks[["ID", "Decision", "Owner", "Status", "Progress"]]

            df_logs = pd.DataFrame(self.audit_trail) if self.audit_trail else pd.DataFrame(columns=["Time", "Agent", "Action"])
            return df_tasks, df_logs

        except Exception as e:
            return pd.DataFrame([{"Error": str(e)}]), pd.DataFrame([])

    def reset(self):
        self.registry, self.audit_trail = [], []
        return pd.DataFrame(columns=["ID", "Decision", "Owner", "Status", "Progress"]), pd.DataFrame(columns=["Time", "Agent", "Action"])

intel = MeetingIntelligenceSystem()

<h3>
  <font face="Arial" color="#FFE3EC">
    <b>Interactive Dashboard</b>
  </font>
</h3>

Finally, we build a Gradio Dashboard. This allows you to paste a meeting transcript into a text box and watch the **"AI Agents"** extract tasks and update their status in real-time.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(primary_hue="slate")) as demo:
    gr.Markdown("# Enterprise Meeting Intelligence")
    gr.Markdown("✔ **Action Extraction** | ✔ **Dependency Mapping** | ✔ **Autonomous Follow-up**")

    with gr.Row():
        with gr.Column(scale=4):
            meeting_input = gr.Textbox(label="Meeting Transcript / Executive Notes", lines=3, placeholder="Type decisions here...")
            with gr.Row():
                run_btn = gr.Button("RUN AUTONOMOUS CYCLE", variant="primary")
                clear_btn = gr.Button("RESET & START NEW SESSION")

    gr.Markdown("### 📊 Live Task Lifecycle")
    task_table = gr.Dataframe(headers=["ID", "Decision", "Owner", "Status", "Progress"], interactive=False)

    gr.Markdown("### 📜 System Audit Trail (Explainable AI)")
    log_table = gr.Dataframe(headers=["Time", "Agent", "Action"], interactive=False)

    # Define button actions
    run_btn.click(intel.process_cycle, inputs=meeting_input, outputs=[task_table, log_table])
    clear_btn.click(intel.reset, outputs=[task_table, log_table])

demo.launch(debug=True, show_error=True)

/tmp/ipykernel_7016/3977035092.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="slate")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://288b93735bfc7ecf07.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<h3>
  <font face="Arial" color="#FFE3EC">
    <b>The Future of Meeting Intelligence</b>
  </font>
</h3>

This Autonomous Workflow Engine demonstrates how generative AI moves beyond simple chat and into active execution.

By combining LLM extraction with a logic-based state machine, we’ve built a system that:

* **Eliminates Human Error:** No more manually transcribing "who said what."

* **Enforces Logic:** Automatically stalls tasks that are missing their prerequisites.

* **Provides Transparency:** The Audit Trail ensures every automated decision is **"explainable"**, showing exactly which AI agent moved a task forward and why.